In [0]:
# Databricks notebook source
# COMMAND ----------

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

SOURCE_MODEL_TABLE = "sandbox.z_jungryo_lee.wls_model_summary_with_corr"
SOURCE_COEF_TABLE = "sandbox.z_jungryo_lee.wls_coef_summary_with_corr"
SOURCE_NETWORK_EDGE_TABLE = "sandbox.z_jungryo_lee.wls_network_edges_with_corr"

MODEL_TABLE = "sandbox.t_online_voc_analysis.voc_wls_model_summary"
COEF_TABLE = "sandbox.t_online_voc_analysis.voc_wls_coef_summary"
NETWORK_EDGE_TABLE = "sandbox.t_online_voc_analysis.voc_wls_network_edges"

DRIVER_PVALUE_THRESHOLD = 0.1
DRIVER_COEF_THRESHOLD = 0.1

MODEL_EXPORT_COLUMNS = [
    "y_feature",
    "y_obs",
    "r_squared",
    "adj_r_squared",
    "f_statistic",
    "prob_f",
    "log_likelihood",
    "aic",
    "bic",
    "cond_no",
    "group_dim",
    "group_key",
    "data_created_dt",
]

COEF_EXPORT_COLUMNS = [
    "y_feature", "x_feature", "coef", "p_value", "t_value",
    "x_obs", "group_dim", "group_key", "abs_coef",
    "driver_rank", "is_driver", "data_created_dt", "weighted_corr"
]

NETWORK_EDGE_EXPORT_COLUMNS = [
    "group_dim", "group_key", "y_feature", "x_feature",
    "coef_sign", "corr_abs", "edge_weight",
    "r_squared", "y_obs", "is_significant", "data_created_dt",
]

print("[CONFIG]")
print(f"  SOURCE_MODEL_TABLE        = {SOURCE_MODEL_TABLE}")
print(f"  SOURCE_COEF_TABLE         = {SOURCE_COEF_TABLE}")
print(f"  SOURCE_NETWORK_EDGE_TABLE = {SOURCE_NETWORK_EDGE_TABLE}")
print(f"  MODEL_TABLE               = {MODEL_TABLE}")
print(f"  COEF_TABLE                = {COEF_TABLE}")
print(f"  NETWORK_EDGE_TABLE        = {NETWORK_EDGE_TABLE}")

# COMMAND ----------

def with_missing_columns(df, column_types):
    for col_name, col_type in column_types.items():
        if col_name not in df.columns:
            df = df.withColumn(col_name, F.lit(None).cast(col_type))
    return df

# COMMAND ----------

# DBTITLE 1,1) Source Load
src_model_df = spark.table(SOURCE_MODEL_TABLE)
src_coef_df = spark.table(SOURCE_COEF_TABLE)
src_network_edge_df = spark.table(SOURCE_NETWORK_EDGE_TABLE)

print("[SOURCE COUNTS]")
print(f"  model rows        = {src_model_df.count():,}")
print(f"  coef rows         = {src_coef_df.count():,}")
print(f"  network edge rows = {src_network_edge_df.count():,}")

# COMMAND ----------

# DBTITLE 1,2) Model Table Transform
# WLS_03 수정본 기준 source schema:
# y_feature, y_obs, r_squared, adj_r_squared, f_statistic, prob_f,
# log_likelihood, aic, bic, cond_no, group_dim, group_key
# data_created_dt가 없으면 publish 시점 timestamp를 부여한다.
src_model_df = with_missing_columns(
    src_model_df,
    {
        "y_feature": "string",
        "y_obs": "bigint",
        "r_squared": "double",
        "adj_r_squared": "double",
        "f_statistic": "double",
        "prob_f": "double",
        "log_likelihood": "double",
        "aic": "double",
        "bic": "double",
        "cond_no": "double",
        "group_dim": "string",
        "group_key": "string",
        "data_created_dt": "timestamp",
    }
)

model_df = (
    src_model_df
    .withColumn("y_obs", F.col("y_obs").cast("bigint"))
    .withColumn("r_squared", F.col("r_squared").cast("double"))
    .withColumn("adj_r_squared", F.col("adj_r_squared").cast("double"))
    .withColumn("f_statistic", F.col("f_statistic").cast("double"))
    .withColumn("prob_f", F.col("prob_f").cast("double"))
    .withColumn("log_likelihood", F.col("log_likelihood").cast("double"))
    .withColumn("aic", F.col("aic").cast("double"))
    .withColumn("bic", F.col("bic").cast("double"))
    .withColumn("cond_no", F.col("cond_no").cast("double"))
    .withColumn("group_dim", F.col("group_dim").cast("string"))
    .withColumn("group_key", F.col("group_key").cast("string"))
    .withColumn("data_created_dt", F.coalesce(F.col("data_created_dt"), F.current_timestamp()))
    .select(*MODEL_EXPORT_COLUMNS)
)

# COMMAND ----------

# DBTITLE 1,3) Coef Table Transform
# coef source는 기존 구조를 유지하되, WLS_03 수정본의 group_dim/group_key를 우선 사용한다.
src_coef_df = with_missing_columns(
    src_coef_df,
    {
        "group_dim": "string",
        "group_key": "string",
        "variable1": "string",
        "variable1_value": "string",
        "t_value": "double",
        "x_obs": "bigint",
        "abs_coef": "double",
        "driver_rank": "bigint",
        "is_driver": "int",
        "data_created_dt": "timestamp",
        "weighted_corr": "double",
    }
)

coef_base_df = (
    src_coef_df
    .withColumn("group_dim", F.coalesce(F.col("group_dim"), F.col("variable1")))
    .withColumn("group_key", F.coalesce(F.col("group_key").cast("string"), F.col("variable1_value").cast("string")))
    .withColumn("group_key", F.col("group_key").cast("string"))
    .withColumn("coef", F.col("coef").cast("double"))
    .withColumn("p_value", F.col("p_value").cast("double"))
    .withColumn("t_value", F.col("t_value").cast("double"))
    .withColumn("x_obs", F.col("x_obs").cast("bigint"))
    .withColumn("weighted_corr", F.col("weighted_corr").cast("double"))
    .withColumn("abs_coef", F.coalesce(F.col("abs_coef").cast("double"), F.abs(F.col("coef"))))
    .withColumn("data_created_dt", F.coalesce(F.col("data_created_dt"), F.current_timestamp()))
)

driver_window = Window.partitionBy(
    "group_dim", "group_key", "y_feature"
).orderBy(F.col("abs_coef").desc(), F.col("x_feature").asc())

coef_df = (
    coef_base_df
    .withColumn(
        "driver_rank",
        F.coalesce(F.col("driver_rank"), F.row_number().over(driver_window))
    )
    .withColumn(
        "is_driver",
        F.coalesce(
            F.col("is_driver"),
            F.when(
                (F.col("p_value") < F.lit(DRIVER_PVALUE_THRESHOLD)) &
                (F.col("abs_coef") >= F.lit(DRIVER_COEF_THRESHOLD)),
                F.lit(1)
            ).otherwise(F.lit(0))
        )
    )
    .select(*COEF_EXPORT_COLUMNS)
)

# COMMAND ----------

# DBTITLE 1,4) Network Edge Transform
# network edge는 별도 생성된 source 테이블을 그대로 publish schema에 맞춰 가져온다.
src_network_edge_df = with_missing_columns(
    src_network_edge_df,
    {
        "group_dim": "string",
        "group_key": "string",
        "variable1": "string",
        "variable1_value": "string",
        "r_squared": "double",
        "r2": "double",
        "y_obs": "bigint",
        "n_obs": "bigint",
        "data_created_dt": "timestamp",
    }
)

network_edge_df = (
    src_network_edge_df
    .withColumn("group_dim", F.coalesce(F.col("group_dim"), F.col("variable1")))
    .withColumn("group_key", F.coalesce(F.col("group_key").cast("string"), F.col("variable1_value").cast("string")))
    .withColumn("r_squared", F.coalesce(F.col("r_squared"), F.col("r2")))
    .withColumn("y_obs", F.coalesce(F.col("y_obs"), F.col("n_obs")))
    .withColumn("group_key", F.col("group_key").cast("string"))
    .withColumn("data_created_dt", F.coalesce(F.col("data_created_dt"), F.current_timestamp()))
    .select(*NETWORK_EDGE_EXPORT_COLUMNS)
)

# COMMAND ----------

# DBTITLE 1,5) Save Final Tables
model_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(MODEL_TABLE)
coef_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(COEF_TABLE)
network_edge_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(NETWORK_EDGE_TABLE)

print("[SAVE DONE]")
print(f"  {MODEL_TABLE}        = {model_df.count():,}")
print(f"  {COEF_TABLE}         = {coef_df.count():,}")
print(f"  {NETWORK_EDGE_TABLE} = {network_edge_df.count():,}")

# COMMAND ----------

# DBTITLE 1,6) Verify
print("[MODEL_TABLE]")
display(spark.table(MODEL_TABLE).limit(10))

print("[COEF_TABLE]")
display(spark.table(COEF_TABLE).limit(10))

print("[NETWORK_EDGE_TABLE]")
display(spark.table(NETWORK_EDGE_TABLE).limit(10))